# CineEmbed — 05 Results (MVP scope)

Generates the **preliminary intermediate-report results table** from `artifacts/eval/results.json`:
- Sklearn baselines: KMeans-raw, PCA+KMeans
- AE family: vanilla_ae_z64, ae_z64
- W1 ablation: ae_z64_w1
- DEC: dec_z64_k21

Total: 6 rows × 3 axes (genre/decade/lang) × 2 metrics (NMI/ARI) = 36 cells.

**Deferred for final report**: 16 more runs (3 VAE + 8 DEC + 2 modality ablations + 2 more AE z values + optional W4) plus full UMAP figures and linear probing.

In [ ]:
import os, sys, json
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Auth: try GitHub token from Colab Secrets (for private repos), fall back to public URL
    try:
        from google.colab import userdata
        _token = userdata.get('GITHUB_TOKEN')
        REPO_URL = f"https://{_token}@github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git"
    except Exception:
        REPO_URL = "https://github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git"
    REPO_ROOT = Path('/content/cineembed-repo')
    ARTIFACTS = Path('/content/drive/MyDrive/CineEmbed/artifacts')

    if not REPO_ROOT.exists():
        get_ipython().system(f'git clone {REPO_URL} {REPO_ROOT}')
    else:
        # Pull latest if you've pushed updates from local
        get_ipython().system(f'cd {REPO_ROOT} && git pull -q')

    get_ipython().system(f'pip install -e {REPO_ROOT} -q')
else:
    REPO_ROOT = Path('..').resolve()
    ARTIFACTS = REPO_ROOT / 'artifacts'

sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import torch

from cineembed import data, backbone, heads, losses, train, eval as cev

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

X, feature_names = data.load_feature_matrix(ARTIFACTS / 'feature_matrix.npz')
labels = data.get_labels(ARTIFACTS / 'movies_eda_final.csv')
block_slices = data.get_block_indices(feature_names)
has_bio = X[:, block_slices['director'].start + 64].clone()
BLOCK_DIMS = {b: (slc.stop - slc.start) for b, slc in block_slices.items()}
PROJ_DIMS = backbone.DEFAULT_PROJ_DIMS
print(f"Loaded {X.shape}; results dir: {ARTIFACTS / 'eval'}")

In [ ]:
import time
from sklearn.cluster import KMeans as SK_KMeans
from sklearn.decomposition import PCA

# Run 1: KMeans on raw 564-dim
print(f"[{time.strftime('%H:%M:%S')}] Computing KMeans on raw 564-dim...")
c_raw = SK_KMeans(n_clusters=21, n_init=20, random_state=42).fit_predict(X.numpy())
m_raw = cev.evaluate_run(c_raw, labels)
m_raw['run_name'] = 'kmeans_raw_k21'
m_raw['z_dim'] = 564

# Run 2: PCA + KMeans
print(f"[{time.strftime('%H:%M:%S')}] Computing PCA(n=64) then KMeans...")
pca = PCA(n_components=64, random_state=42).fit(X.numpy())
X_pca = pca.transform(X.numpy())
c_pca = SK_KMeans(n_clusters=21, n_init=20, random_state=42).fit_predict(X_pca)
m_pca = cev.evaluate_run(c_pca, labels)
m_pca['run_name'] = 'pca_kmeans_k21'
m_pca['z_dim'] = 64

results_path = ARTIFACTS / 'eval' / 'results.json'
existing = json.loads(results_path.read_text()) if results_path.exists() else {}
existing[m_raw['run_name']] = m_raw
existing[m_pca['run_name']] = m_pca
results_path.write_text(json.dumps(existing, indent=2))

print(f"[{time.strftime('%H:%M:%S')}] Baselines: kmeans_raw NMI(genre)={m_raw['genre_nmi']:.3f}, "
      f"pca_kmeans NMI(genre)={m_pca['genre_nmi']:.3f}")


In [ ]:
import pandas as pd
results = json.loads(results_path.read_text())

EXPECTED_RUNS = ['kmeans_raw_k21', 'pca_kmeans_k21', 'vanilla_ae_z64', 'ae_z64',
                 'ae_z64_w1', 'dec_z64_k21']
rows = []
for name in EXPECTED_RUNS:
    if name not in results:
        rows.append({'run': name, 'status': 'MISSING'})
        continue
    m = results[name]
    rows.append({
        'run': name,
        'z': m.get('z_dim', '-'),
        'k': m.get('k', '-'),
        'genre_NMI': round(m.get('genre_nmi', float('nan')), 3),
        'genre_ARI': round(m.get('genre_ari', float('nan')), 3),
        'decade_NMI': round(m.get('decade_nmi', float('nan')), 3),
        'decade_ARI': round(m.get('decade_ari', float('nan')), 3),
        'lang_NMI': round(m.get('lang_nmi', float('nan')), 3),
        'lang_ARI': round(m.get('lang_ari', float('nan')), 3),
    })
df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))
df_results.to_csv(ARTIFACTS / 'eval' / 'results_table_mvp.csv', index=False)
print(f"\nSaved to {ARTIFACTS / 'eval' / 'results_table_mvp.csv'}")

In [ ]:
# Quick narrative for the intermediate report
print("=" * 72)
print("MVP Results Narrative")
print("=" * 72)

baseline_keys = ['kmeans_raw_k21', 'pca_kmeans_k21', 'vanilla_ae_z64']
deep_keys = ['ae_z64', 'ae_z64_w1', 'dec_z64_k21']

best_baseline = max(
    (results.get(k, {}).get('genre_nmi', 0) for k in baseline_keys),
    default=0,
)
best_deep = max(
    (results.get(k, {}).get('genre_nmi', 0) for k in deep_keys),
    default=0,
)

print(f"Best baseline NMI(genre):   {best_baseline:.3f}")
print(f"Best deep NMI(genre):       {best_deep:.3f}")
if best_baseline > 0:
    rel_gain = (best_deep / best_baseline - 1) * 100
    print(f"Relative improvement:       {rel_gain:+.1f}%")
print(f"Absolute floor (>= 0.15):    {'PASS' if best_deep >= 0.15 else 'NEEDS DISCUSSION'}")
print(f"10% relative gain target:   {'PASS' if best_deep >= best_baseline * 1.10 else 'NEEDS DISCUSSION'}")

# W1 vs W2 ablation (validates D3 weighting decision)
w1_nmi = results.get('ae_z64_w1', {}).get('genre_nmi', 0)
w2_nmi = results.get('ae_z64', {}).get('genre_nmi', 0)
if w1_nmi > 0 and w2_nmi > 0:
    print(f"\nW2 vs W1 ablation (z=64 AE):")
    print(f"  W2 (inverse-variance): NMI={w2_nmi:.3f}")
    print(f"  W1 (uniform):          NMI={w1_nmi:.3f}")
    print(f"  Validates W2:          {'YES' if w2_nmi > w1_nmi * 1.05 else 'INCONCLUSIVE'}")

# Multi-modal vs vanilla architecture comparison
mm_nmi = results.get('ae_z64', {}).get('genre_nmi', 0)
van_nmi = results.get('vanilla_ae_z64', {}).get('genre_nmi', 0)
if mm_nmi > 0 and van_nmi > 0:
    print(f"\nMulti-modal vs vanilla architecture (z=64):")
    print(f"  Multi-modal AE:  NMI={mm_nmi:.3f}")
    print(f"  Vanilla concat:  NMI={van_nmi:.3f}")
    print(f"  Validates D1:    {'YES' if mm_nmi > van_nmi * 1.05 else 'INCONCLUSIVE'}")

## Final report extension (post-intermediate)

After the intermediate report, this notebook will be extended with:
- Full 3×3×3 results table (3 model families × 3 z dims × 3 axes)
- Linear probing accuracy on z=64 frozen latents
- Modality ablation deltas (F1: text removed, F2: director removed)
- 12 main report UMAP figures (best of each model family × 3 axes + 3 baselines × genre)
- Reproducibility audit (two-run MD5 check)